# Première partie : L'Analyse "Macro" (Le Marché Global) 

In [0]:
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.types import *

# On récupére le json source qui est sur le bucket jedha
s3_url = "s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json"

#Lecture du fichier JSON
df_steam = spark.read.json(s3_url)

# affichage de la structure des données pour avoir les différents noeuds
df_steam.printSchema()

root
 |-- data: struct (nullable = true)
 |    |-- appid: long (nullable = true)
 |    |-- categories: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- ccu: long (nullable = true)
 |    |-- developer: string (nullable = true)
 |    |-- discount: string (nullable = true)
 |    |-- genre: string (nullable = true)
 |    |-- header_image: string (nullable = true)
 |    |-- initialprice: string (nullable = true)
 |    |-- languages: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- negative: long (nullable = true)
 |    |-- owners: string (nullable = true)
 |    |-- platforms: struct (nullable = true)
 |    |    |-- linux: boolean (nullable = true)
 |    |    |-- mac: boolean (nullable = true)
 |    |    |-- windows: boolean (nullable = true)
 |    |-- positive: long (nullable = true)
 |    |-- price: string (nullable = true)
 |    |-- publisher: string (nullable = true)
 |    |-- release_date: string (nullable = true)
 |    |-

In [0]:

# à partir du prenmier noeud 'data' on crée des colonnes à partir des enfants
# on met tout à plat
df_flat = df_steam.select(
    F.col("id"),
    F.col("data.appid").alias("appid"),
    F.col("data.name").alias("name"),
    F.col("data.publisher").alias("publisher"),
    F.col("data.developer").alias("developer"),
    F.col("data.genre").alias("genre"),
    F.col("data.initialprice").alias("initialprice"),
    F.col("data.price").alias("price"),
    F.col("data.discount").alias("discount"),
    F.col("data.release_date").alias("release_date"),
    F.col("data.required_age").alias("required_age"),
    F.col("data.languages").alias("languages"),
    F.col("data.positive").alias("positive"),
    F.col("data.negative").alias("negative"),
    F.col("data.platforms").alias("platforms") 
)

# aperçu des données
display(df_flat.head(10))

id,appid,name,publisher,developer,genre,initialprice,price,discount,release_date,required_age,languages,positive,negative,platforms
10,10,Counter-Strike,Valve,Valve,Action,999,999,0,2000/11/1,0,"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean",201215,5199,"List(true, true, true)"
1000000,1000000,ASCENXION,PsychoFlux Entertainment,IndigoBlue Game Studio,"Action, Adventure, Indie",999,999,0,2021/05/14,0,"English, Korean, Simplified Chinese",27,5,"List(false, false, true)"
1000010,1000010,Crown Trick,"Team17, NEXT Studios",NEXT Studios,"Adventure, Indie, RPG, Strategy",1999,599,70,2020/10/16,0,"Simplified Chinese, English, Japanese, Traditional Chinese, French, German, Spanish - Spain, Russian, Portuguese - Brazil",4032,646,"List(false, false, true)"
1000030,1000030,"Cook, Serve, Delicious! 3?!",Vertigo Gaming Inc.,Vertigo Gaming Inc.,"Action, Indie, Simulation, Strategy",1999,1999,0,2020/10/14,0,English,1575,115,"List(false, true, true)"
1000040,1000040,细胞战争,DoubleC Games,DoubleC Games,"Action, Casual, Indie, Simulation",199,199,0,2019/03/30,0,Simplified Chinese,0,1,"List(false, false, true)"
1000080,1000080,Zengeon,2P Games,IndieLeague Studio,"Action, Adventure, Indie, RPG",1999,799,60,2019/06/24,0,"Simplified Chinese, English, Traditional Chinese, Japanese, Korean",1018,462,"List(false, true, true)"
1000100,1000100,干支セトラ 陽ノ卷｜干支etc. 陽之卷,Starship Studio,七月九日,"Adventure, Indie, RPG, Strategy",1299,1299,0,2019/01/24,0,"Japanese, Simplified Chinese, Traditional Chinese",18,6,"List(false, false, true)"
1000110,1000110,Jumping Master(跳跳大咖),重庆环游者网络科技,重庆环游者网络科技,"Action, Adventure, Casual, Free to Play, Massively Multiplayer",0,0,0,2019/04/8,0,"English, Simplified Chinese, Traditional Chinese",50,34,"List(false, false, true)"
1000130,1000130,Cube Defender,Simon Codrington,Simon Codrington,"Casual, Indie",299,299,0,2019/01/6,0,English,6,0,"List(false, true, true)"
1000280,1000280,Tower of Origin2-Worm's Nest,Villain Role,Villain Role,"Indie, RPG",1399,1399,0,2021/09/9,0,"English, Simplified Chinese, Traditional Chinese",32,12,"List(false, false, true)"


In [0]:
# QUESTION 1 : Quel éditeur a sorti le plus de jeux ? 
# Groupe par éditeur, on compte, et on trie par ordre décroissant (on exlue les éditeur 'vide')
df_top_publishers = df_flat.groupBy("publisher") \
                           .count() \
                           .sort(F.col("count").desc()) \
                           .filter("publisher IS NOT NULL") 

df_top_publishers.show(10, truncate=False)

+---------------+-----+
|publisher      |count|
+---------------+-----+
|Big Fish Games |422  |
|8floor         |202  |
|SEGA           |165  |
|Strategy First |151  |
|Square Enix    |141  |
|Choice of Games|140  |
|               |132  |
|Sekai Project  |132  |
|HH-Games       |132  |
|Ubisoft        |127  |
+---------------+-----+
only showing top 10 rows


réponse : l'éditeur Big Fish Games avec pas moins de 422 jeux.

In [0]:
# # QUESTION 2 : Quels sont les jeux les mieux notés ? 

# calcule du total des avis (positifs + négatifs)
df_with_total = df_flat.withColumn("total_avis", F.col("positive") + F.col("negative"))

# Taux de satisfaction en % 
df_with_ratio = df_with_total.withColumn(
    "taux_satisfaction", 
    F.round((F.col("positive") / F.col("total_avis")) * 100, 2)
)

# J"ai été obligé de garder uniquement les jeux qui ont le plus d'avis (+ de 1 000)
df_filtered = df_with_ratio.filter("positive > 1000")

#Trie des colonnes à garder et trie de l'affichage
df_top_rated_final = df_filtered.select("name", "positive", "negative", "taux_satisfaction") \
                                .sort(F.col("taux_satisfaction").desc())

df_top_rated_final.show(10, truncate=False)

+-----------------------------------+--------+--------+-----------------+
|name                               |positive|negative|taux_satisfaction|
+-----------------------------------+--------+--------+-----------------+
|Aventura Copilului Albastru și Urât|2203    |14      |99.37            |
|Aseprite                           |11823   |80      |99.33            |
|Patrick's Parabox                  |1489    |11      |99.27            |
|A Short Hike                       |11645   |87      |99.26            |
|Monster Prom 3: Monster Roadtrip   |1052    |8       |99.25            |
|Senren＊Banka                      |10593   |84      |99.21            |
|CULTIC                             |2021    |16      |99.21            |
|Ib                                 |1814    |15      |99.18            |
|星空列车与白的旅行                 |1549    |14      |99.1             |
|planetarian HD                     |1952    |20      |98.99            |
+-----------------------------------+--------+--

réponse : Le jeu le mieux noté est 'Aventura Copilului Albastru și Urât' 

In [0]:
# QUESTION 3 : Évolution des sorties et impact Covid

# Conversion des champs date pour extraire l'année
# utilisation de la fonction try_to_date car certaines données ne sont pas au format année/mois/jour
df_years_fixed = df_flat.withColumn("date_parsed", F.try_to_date(F.col("release_date"), "yyyy/MM/d")) \
                        .withColumn("release_year", F.year(F.col("date_parsed")))

# 2. Groupement et décompte de 2010 à 2023
df_trends = df_years_fixed.groupBy("release_year") \
                           .count() \
                           .sort("release_year") \
                           .filter("release_year IS NOT NULL AND release_year >= 2010 AND release_year <= 2026")

df_trends.show(20)

+------------+-----+
|release_year|count|
+------------+-----+
|        2010|  281|
|        2011|  267|
|        2012|  344|
|        2013|  469|
|        2014| 1550|
|        2015| 2566|
|        2016| 4176|
|        2017| 6006|
|        2018| 7663|
|        2019| 6949|
|        2020| 8287|
|        2021| 8805|
|        2022| 7451|
+------------+-----+



In [0]:
# lancement de la visualisation par pyspark
display(df_trends)

release_year,count
2010,281
2011,267
2012,344
2013,469
2014,1550
2015,2566
2016,4176
2017,6006
2018,7663
2019,6949


Databricks visualization. Run in Databricks to view.

![Evolution des sorties de 2010 à 2022](screen_shot/graphique_question3_evolution_des_sortie_impacte_covid.png)

réponse : <br>
La courbe de croissance des ventes a subi un recule en 2019 et 2022 avec un volume de vente à la baisse.<br>
En prenant comme référence 2018, qui est sur un progression de 21 % par rapport à 2017.<br>
On voit que la moyenne des 4 années suivante augmente seulement de 2.7%, ce qui est clairement la période et les conséquences du Covid


In [0]:
# QUESTION 4 : Comment les prix sont-ils distribués ? Y a-t-il beaucoup de jeux avec une réduction ? 

# cast du prix chiffre (en divisant par 100 pour avoir des euros)
df_simple_price = df_flat.withColumn("prix_euro", F.round(F.col("price").cast("float") / 100, 2))

# filtre pour tirer les promotions (valide et != 0)
df_promos = df_simple_price.withColumn(
    "is_discounted", 
    F.when((F.col("discount") != "0") & (F.col("discount").isNotNull()), "En Promotion").otherwise("Plein Tarif")
)

df_promos.groupBy("is_discounted").count().show()
print("Seulement 4.52 pourcent des jeux sont en promotion")

# 3. Préparation de la distribution des prix (on filtre le catalogue pour les jeux entre 0€ et 60€ pour éviter les valeurs extrêmes)
df_distribution_clean = df_simple_price.filter("prix_euro IS NOT NULL AND prix_euro <= 60") \
                                       .select("prix_euro")

display(df_distribution_clean)

+-------------+-----+
|is_discounted|count|
+-------------+-----+
|  Plein Tarif|53173|
| En Promotion| 2518|
+-------------+-----+



prix_euro
9.99
9.99
5.99
19.99
1.99
7.99
12.99
0.0
2.99
13.99


Databricks visualization. Run in Databricks to view.

![Prix des jeux proposés](screen_shot/graphique_question4_reduction_et_prix.png)

Réponse : <br>
Steam est totalement positionné sur des jeux à coût faible (moins de 10 euros) avec environ 80% de son volume


In [0]:
# QUESTION 5 : Quelles sont les langues les plus représentées ? 

# Création de x nouvelles lignes en fonction du split des langues
df_lang_exploded = df_flat.select(F.explode(F.split(F.col("languages"), ", ")).alias("langue"))

#maintenant on regroupe pour compter
df_lang_exploded.groupBy("langue") \
                 .count() \
                 .sort(F.col("count").desc()) \
                 .show(10)

+-------------------+-----+
|             langue|count|
+-------------------+-----+
|            English|55116|
|             German|14019|
|             French|13426|
|            Russian|12922|
| Simplified Chinese|12782|
|    Spanish - Spain|12233|
|           Japanese|10368|
|            Italian| 9304|
|Portuguese - Brazil| 6750|
|             Korean| 6599|
+-------------------+-----+
only showing top 10 rows


Réponse : <br>
L'anglais est incontestablement la langue la plus représentée.

In [0]:
# QUESTION 6 : Jeux interdits aux moins de 18 ans

# cast de l'âge en nombre
df_simple_age = df_flat.withColumn("age", F.col("required_age").try_cast("int"))

# méthode de trie : si l'âge est égal à 0 OU supérieur ou égal à 18 -> plus de 18 ans
df_grouped_age = df_simple_age.withColumn(
    "categorie_age",
    F.when((F.col("age") == 0) | (F.col("age") >= 18), "Plus de 18 ans ")
     .otherwise("Moins de 18 ans")
)

df_grouped_age.groupBy("categorie_age").count().show()

+---------------+-----+
|  categorie_age|count|
+---------------+-----+
|Plus de 18 ans |55259|
|Moins de 18 ans|  432|
+---------------+-----+



Réponse : <br>
Avec plus de 99% de ses ventes, Steam se position sur un public d'adulte. 
